# 🔬 Particle Physics Event Classifier — Training Notebook

**GPU-accelerated training on Google Colab (T4)**

> **Before running:** `Runtime → Change runtime type → T4 GPU`

| Model | T4 Time (500k events) |
|---|---|
| MLP (100 epochs) | ~8–15 min |
| XGBoost BDT | ~5–10 min |
| LightGBM BDT | ~3–6 min |
| Optuna HPO (50 trials) | ~60–90 min |

Everything saves to your **Google Drive** — safe across session disconnects.

## ⚙️ 0. GPU Check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    # Just show GPU name line
    for line in result.stdout.splitlines():
        if 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line:
            print('GPU:', line.strip())
            break
    else:
        print(result.stdout[:500])
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 💾 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/particle_physics_classifier'
for folder in ['data/raw', 'data/processed', 'data/cache', 'data/feature_store',
               'models', 'mlruns', 'configs']:
    os.makedirs(f'{DRIVE_ROOT}/{folder}', exist_ok=True)

print(f'Drive root: {DRIVE_ROOT}')

## 📦 2. Clone Repo & Install Dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/rayxxee/Particle-Physics-Classifier.git'
REPO_DIR = '/content/particle_physics_classifier'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Working dir: {os.getcwd()}')

In [ ]:
# Install project deps (torch already on Colab, skip reinstall)
# BUG FIX: install xgboost + lightgbm explicitly (they're in [bdt] optional group)
!pip install -q pandera omegaconf structlog optuna mlflow xgboost lightgbm PyYAML
!pip install -q -e . --no-deps 2>&1 | tail -3

# Verify imports
import torch, numpy, pandas, optuna, mlflow, xgboost, lightgbm
print('All imports OK')
print(f'  torch   {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'  optuna  {optuna.__version__}')
print(f'  mlflow  {mlflow.__version__}')
print(f'  xgboost {xgboost.__version__}')

## 📥 3. Download HIGGS Dataset
Downloaded once to Drive (7 GB). All future sessions skip this step.

In [ ]:
import os

RAW_DIR   = f'{DRIVE_ROOT}/data/raw'
HIGGS_CSV = f'{RAW_DIR}/HIGGS.csv'
HIGGS_GZ  = f'{RAW_DIR}/HIGGS.csv.gz'
HIGGS_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'

if os.path.exists(HIGGS_CSV):
    size_gb = os.path.getsize(HIGGS_CSV) / 1e9
    print(f'✅  HIGGS.csv already on Drive ({size_gb:.1f} GB) — skipping download')
elif os.path.exists(HIGGS_GZ):
    print('Decompressing .gz (already downloaded)...')
    os.system(f'gunzip -k {HIGGS_GZ}')
    print('Done.')
else:
    print('Downloading HIGGS (~2.6 GB compressed, ~7 GB uncompressed)...')
    os.system(f'wget -q --show-progress -O {HIGGS_GZ} {HIGGS_URL}')
    print('Decompressing...')
    os.system(f'gunzip -k {HIGGS_GZ}')
    print('✅  Done.')

print(f'CSV size: {os.path.getsize(HIGGS_CSV)/1e9:.2f} GB')

## 🔧 4. Configuration — Edit This Cell

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
N_SAMPLES    = 500_000   # 100_000 for quick test | 500_000 recommended | None = all 11M
TRAIN_MODEL  = 'mlp'     # 'mlp' | 'bdt' | 'hpo' | 'all'
HPO_N_TRIALS = 50        # Optuna trials (if TRAIN_MODEL='hpo' or 'all')
HPO_N_EPOCHS = 30        # Epochs per HPO trial
# ──────────────────────────────────────────────────────────────────────────────

HIGGS_CSV      = f'{DRIVE_ROOT}/data/raw/HIGGS.csv'
PROCESSED_DIR  = f'{DRIVE_ROOT}/data/processed'
CACHE_DIR      = f'{DRIVE_ROOT}/data/cache'
FEATURE_DIR    = f'{DRIVE_ROOT}/data/feature_store'
MODELS_DIR     = f'{DRIVE_ROOT}/models'
MLFLOW_DIR     = f'{DRIVE_ROOT}/mlruns'
BEST_CFG_PATH  = f'{DRIVE_ROOT}/configs/mlp_best.yaml'
OPTUNA_DB      = f'sqlite:///{DRIVE_ROOT}/optuna.db'

import mlflow
mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
mlflow.set_experiment('particle_physics_classifier')

print(f'n_samples   : {N_SAMPLES:,}' if N_SAMPLES else 'n_samples   : ALL (11M)')
print(f'model       : {TRAIN_MODEL}')
print(f'mlflow      : {MLFLOW_DIR}')

## 🔄 5. ETL Pipeline

In [ ]:
from src.ingestion.etl_pipeline import ETLConfig, ETLPipeline

etl_cfg = ETLConfig(
    raw_filepath     = HIGGS_CSV,
    n_samples        = N_SAMPLES,
    processed_dir    = PROCESSED_DIR,
    cache_dir        = CACHE_DIR,
    use_reader_cache = True,   # Cache the CSV parse — big speedup on reruns
)

splits = ETLPipeline(etl_cfg).run()

print(f'\n✅ ETL complete  version={splits.version}')
print(f'   train={splits.n_train:,}  val={splits.n_val:,}  test={splits.n_test:,}  features={splits.n_features}')

## 🧮 6. Feature Engineering

In [ ]:
from src.features.feature_store import FeatureConfig, FeatureStore

feat_cfg = FeatureConfig(
    include_low_level  = True,
    include_dataset_hl = True,
    include_derived_hl = True,  # adds ΔR, m_T, H_T, centrality, MET significance
)

store = FeatureStore(feat_cfg, cache_dir=FEATURE_DIR)
store.build(splits)

X_train, y_train = store.load('train')
X_val,   y_val   = store.load('val')
X_test,  y_test  = store.load('test')

print(f'\n✅ Features ready')
print(f'   train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}')
print(f'   cols : {list(X_train.columns[:5])} ...')

## 🧠 7a. Train MLP Baseline

In [ ]:
if TRAIN_MODEL in ('mlp', 'all'):
    import contextlib
    from sklearn.metrics import roc_auc_score
    from src.models.mlp.config import MLPConfig
    from src.models.mlp.model  import MLPModel

    mlp_cfg = MLPConfig(
        input_dim       = X_train.shape[1],
        hidden_dims     = [512, 256, 128, 64],
        dropout_rates   = [0.3, 0.3, 0.2, 0.0],
        batch_norm      = True,
        epochs          = 100,
        batch_size      = 4096,
        learning_rate   = 1e-3,
        weight_decay    = 1e-4,
        scheduler_name  = 'cosine_annealing',
        early_stopping  = True,
        early_stopping_patience = 10,
        mixed_precision = True,   # AMP gives ~40% speedup on T4
        seed = 42,
    )

    mlp = MLPModel(mlp_cfg)

    with mlflow.start_run(run_name='mlp_baseline'):
        mlflow.log_params(mlp_cfg.to_dict())

        # BUG FIX: fit() now returns best_val_auc float (not history dict)
        best_val_auc = mlp.fit(
            X_train.values, y_train.values,
            X_val.values,   y_val.values,
        )

        test_auc = roc_auc_score(y_test.values, mlp.predict_proba(X_test.values))

        mlflow.log_metric('best_val_auc', best_val_auc)
        mlflow.log_metric('test_auc',     test_auc)

    mlp.save(f'{MODELS_DIR}/mlp_baseline')

    print(f'\n✅ MLP complete')
    print(f'   Best val AUC : {best_val_auc:.4f}')
    print(f'   Test AUC     : {test_auc:.4f}')
    print(f'   Saved        : {MODELS_DIR}/mlp_baseline')
else:
    print('Skipping MLP (set TRAIN_MODEL="mlp" or "all" to enable)')

## 🌲 7b. Train BDT Baselines (XGBoost + LightGBM)

In [ ]:
if TRAIN_MODEL in ('bdt', 'all'):
    import joblib
    import numpy as np
    import xgboost as xgb
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score

    X_tr = X_train.values.astype('float32')
    y_tr = y_train.values.astype('float32')
    X_v  = X_val.values.astype('float32')
    y_v  = y_val.values.astype('float32')
    X_te = X_test.values.astype('float32')
    y_te = y_test.values.astype('float32')

    # ── XGBoost ──────────────────────────────────────────────────────────────
    print('Training XGBoost...')
    import torch as _torch
    # BUG FIX: XGBoost >=2.0 uses device='cuda', not tree_method='gpu_hist'
    xgb_device = 'cuda' if _torch.cuda.is_available() else 'cpu'
    xgb_model  = xgb.XGBClassifier(
        n_estimators          = 1000,
        max_depth             = 6,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        device                = xgb_device,
        tree_method           = 'hist',
        eval_metric           = 'auc',
        early_stopping_rounds = 20,
        random_state          = 42,
        n_jobs                = -1,
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=100)
    xgb_auc = roc_auc_score(y_te, xgb_model.predict_proba(X_te)[:, 1])
    print(f'✅  XGBoost test AUC: {xgb_auc:.4f}')

    # ── LightGBM ─────────────────────────────────────────────────────────────
    print('\nTraining LightGBM...')
    # BUG FIX: LightGBM GPU requires special build; default to cpu for safety
    lgb_model = lgb.LGBMClassifier(
        n_estimators     = 1000,
        max_depth        = 6,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        num_leaves       = 63,
        random_state     = 42,
        n_jobs           = -1,
    )
    lgb_model.fit(
        X_tr, y_tr,
        eval_set  = [(X_v, y_v)],
        callbacks = [lgb.early_stopping(20), lgb.log_evaluation(100)],
    )
    lgb_auc = roc_auc_score(y_te, lgb_model.predict_proba(X_te)[:, 1])
    print(f'✅  LightGBM test AUC: {lgb_auc:.4f}')

    # Save
    joblib.dump(xgb_model, f'{MODELS_DIR}/xgboost.pkl')
    joblib.dump(lgb_model, f'{MODELS_DIR}/lightgbm.pkl')

    with mlflow.start_run(run_name='bdt_baselines'):
        mlflow.log_metric('xgb_test_auc', xgb_auc)
        mlflow.log_metric('lgb_test_auc', lgb_auc)

    print(f'\nModels saved to {MODELS_DIR}')
else:
    print('Skipping BDT (set TRAIN_MODEL="bdt" or "all" to enable)')

## 🔍 7c. Optuna HPO (MLP Architecture Search)

In [ ]:
if TRAIN_MODEL in ('hpo', 'all'):
    from src.models.mlp.optimizer import HPOConfig, MLPOptimizer
    from src.models.mlp.model     import MLPModel
    from sklearn.metrics import roc_auc_score

    hpo_cfg = HPOConfig(
        n_trials           = HPO_N_TRIALS,
        n_epochs_per_trial = HPO_N_EPOCHS,
        study_name         = 'mlp_hpo',
        storage            = OPTUNA_DB,    # Saved to Drive — survives disconnects!
        save_best_config   = BEST_CFG_PATH,
        n_startup_trials   = 10,
        n_warmup_steps     = 5,
        seed               = 42,
    )

    optimizer = MLPOptimizer(hpo_cfg)
    best_cfg  = optimizer.run(
        X_train.values, y_train.values,
        X_val.values,   y_val.values,
    )

    print(f'\n✅  HPO done  best_val_auc={optimizer.study.best_value:.4f}')
    print(f'   Best params: {optimizer.study.best_params}')
    print(f'   Config saved to {BEST_CFG_PATH}')

    # ── Retrain final model at full epochs with best config
    print('\nRetraining final model with best config (100 epochs)...')
    best_cfg.epochs          = 100
    best_cfg.mixed_precision = True
    hpo_model = MLPModel(best_cfg)

    with mlflow.start_run(run_name='mlp_hpo_final'):
        mlflow.log_params(best_cfg.to_dict())
        best_val = hpo_model.fit(
            X_train.values, y_train.values,
            X_val.values,   y_val.values,
        )
        test_auc = roc_auc_score(y_test.values, hpo_model.predict_proba(X_test.values))
        mlflow.log_metric('test_auc', test_auc)

    hpo_model.save(f'{MODELS_DIR}/mlp_hpo_best')
    print(f'\n✅  Final test AUC: {test_auc:.4f}')

    # ── Importance plot
    import matplotlib.pyplot as plt
    imp = optimizer.importance()
    fig, ax = plt.subplots(figsize=(8, max(3, len(imp) * 0.35)))
    names, vals = zip(*sorted(imp.items(), key=lambda x: x[1]))
    ax.barh(names, vals, color='steelblue')
    ax.set_xlabel('Importance Score')
    ax.set_title('Hyperparameter Importance (Optuna FAnova)')
    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/hpo_importance.png', dpi=150)
    plt.show()
else:
    print('Skipping HPO (set TRAIN_MODEL="hpo" or "all" to enable)')

## 📈 8. Evaluation — ROC Curves & Score Distributions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc as sk_auc, roc_auc_score

# Collect scores from all trained models
results = {}  # name → (scores array, color)

try:
    results['MLP baseline'] = (mlp.predict_proba(X_test.values), '#2196F3')
except NameError:
    pass
try:
    results['MLP (HPO)'] = (hpo_model.predict_proba(X_test.values), '#673AB7')
except NameError:
    pass
try:
    results['XGBoost'] = (xgb_model.predict_proba(X_te)[:, 1], '#4CAF50')
except NameError:
    pass
try:
    results['LightGBM'] = (lgb_model.predict_proba(X_te)[:, 1], '#FF9800')
except NameError:
    pass

y_true = y_test.values

if not results:
    print('No models trained yet — run cells above first')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── ROC curves
    ax = axes[0]
    for name, (scores, color) in results.items():
        fpr, tpr, _ = roc_curve(y_true, scores)
        auc_val = sk_auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name}  AUC={auc_val:.4f}', color=color, lw=2)
    ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
    ax.set_xlabel('False Positive Rate (1 - Specificity)')
    ax.set_ylabel('True Positive Rate (Sensitivity)')
    ax.set_title('ROC Curves')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)

    # ── Score distributions for best model
    ax = axes[1]
    best_name, (best_scores, best_color) = list(results.items())[0]
    ax.hist(best_scores[y_true==0], bins=60, alpha=0.6, label='Background (y=0)',
            color='#F44336', density=True)
    ax.hist(best_scores[y_true==1], bins=60, alpha=0.6, label='Signal (y=1)',
            color='#2196F3', density=True)
    ax.set_xlabel('Classifier Score P(signal)')
    ax.set_ylabel('Density')
    ax.set_title(f'Score Distribution — {best_name}')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.suptitle('Particle Physics Event Classifier', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nPlot saved: {DRIVE_ROOT}/results.png')

## 💾 9. Save Summary

In [ ]:
import json, datetime
from sklearn.metrics import roc_auc_score

summary = {
    'timestamp'  : datetime.datetime.now().isoformat(),
    'n_samples'  : N_SAMPLES,
    'n_features' : int(X_train.shape[1]),
    'models'     : {},
}

print('=== Final Results ===')
for name, (scores, _) in results.items():
    auc_val = roc_auc_score(y_true, scores)
    summary['models'][name] = {'test_auc': round(float(auc_val), 6)}
    print(f'  {name:<20}  AUC = {auc_val:.4f}')

with open(f'{DRIVE_ROOT}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n✅  Summary saved: {DRIVE_ROOT}/training_summary.json')
print(f'    Models folder : {MODELS_DIR}')
print(f'    MLflow runs   : {MLFLOW_DIR}')

---
## 📌 Quick Reference

| What | Where on Drive |
|---|---|
| Trained models | `particle_physics_classifier/models/` |
| Results plot | `particle_physics_classifier/results.png` |
| Best HPO config | `particle_physics_classifier/configs/mlp_best.yaml` |
| Optuna study | `particle_physics_classifier/optuna.db` |
| MLflow runs | `particle_physics_classifier/mlruns/` |
| Summary JSON | `particle_physics_classifier/training_summary.json` |

**On session restart:** Run cells 0→4 (takes ~2 min), then skip to cell 5 (ETL hits cache instantly).